In [2]:
import getpass
import os

endpoint = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT") or input("Endpoint: ").strip()
key = os.getenv("AZURE_DOCUMENT_INTELLIGENCE_KEY") or getpass.getpass("Key: ")

assert endpoint and key, "Set endpoint and key before continuing"

In [3]:
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.core.credentials import AzureKeyCredential

client = DocumentIntelligenceClient(endpoint=endpoint, credential=AzureKeyCredential(key))

invoice_url = "https://raw.githubusercontent.com/Percy04/take_home_zamp_invoice/main/test_data/hard_table_1.png"
poller = client.begin_analyze_document(
    "prebuilt-invoice",
    AnalyzeDocumentRequest(url_source=invoice_url),
)
result = poller.result()

len(result.documents), result.model_id

(1, 'prebuilt-invoice')

In [7]:
from IPython.display import Markdown, display

def value(obj, name, default=None):
    return getattr(obj, name, None) if hasattr(obj, name) else obj.get(name, default)

def clean(text):
    return str(text or "").replace("|", "\\|").replace("\n", "<br>")

if not result.tables:
    print("No tables found")

for index, table in enumerate(result.tables or [], start=1):
    rows = value(table, "row_count", 0)
    cols = value(table, "column_count", 0)
    grid = [["" for _ in range(cols)] for _ in range(rows)]

    for cell in value(table, "cells", []):
        row = value(cell, "row_index")
        col = value(cell, "column_index")
        if row is not None and col is not None:
            grid[row][col] = clean(value(cell, "content", ""))

    lines = [f"### Table {index}", ""]
    lines.append("| " + " | ".join(grid[0]) + " |")
    lines.append("| " + " | ".join(["---"] * cols) + " |")
    for row in grid[1:]:
        lines.append("| " + " | ".join(row) + " |")

    display(Markdown("\n".join(lines)))

### Table 1

| INVOICE NUMBER | INVOICE DATE | PAYMENT DUE DATE | OUR PROJECT NO. | BALANCE DUE |
| --- | --- | --- | --- | --- |
| GALT-009 | Aug 31, 2013 | Sep 30, 2013 | 2012-0001 | $11,812.50 |

### Table 2

|  | Fee Summary |  | Previously Invoiced |  | Current Invoice |  | Remaining |  |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
|  | % | Stipulated | % phase<br>completed | Amount billed | %<br>Complete | Value of<br>completed | Amount<br>Remaining |  |
| 02 Schematic Design | 15.63% | $80,000.00 | 100.00% | $80,000.00 | 0.00% | $0.00 | $0.00 |  |
| 03 Design Development | 23.44% | $120,000.00 | 100.00% | $120,000.00 | 0.00% | $0.00 | $0.00 |  |
| 04 Construction Documents | 29.30% | $150,000.00 | 60.00% | $90,000.00 | 66.00% | $9,000.00 | $51,000.00 |  |
|  | 68.36% | $350,000.00 | 82.86% | $290,000.00 | 2.57% | $9,000.00 | $51,000.00 |  |

### Table 3

|  |  |  |  |  |  |
| --- | --- | --- | --- | --- | --- |
| Date | Employee | Code | Description | Hrs | Extension |
| 06 Contract Administration |  |  |  |  |  |
| 8/1/2013 | DF | Basic Services | Prepare Payout Request Log | 3.00 | $375.00 |
| 8/1/2013 | DF | Basic Services | Preconstruction Meeting with Owner<br>and Contractor | 3.00 | $375.00 |
| 8/1/2013 | DF | Basic Services | Review Procedures for Submittal and<br>Review of Payout Requests with<br>Contractor | 2.00 | $250.00 |
| 8/7/2013 | HR | Basic Services | Project Directory - Construction/test | 1.00 | $250.00 |
|  |  |  | 06 Contract Administration Total: | 9.00 | $1,250.00 |
|  |  |  | Basic Services Sub Total: | 9.00 | $1,250.00 |

### Table 4

| Date | Name | Code | Description | Hrs | Extension |
| --- | --- | --- | --- | --- | --- |
| 02 Schematic Design<br>8/15/2013 HR |  |  |  |  |  |
|  |  | Existing Facilities Survey |  | 6.25 | $1,562.50 |
|  |  |  | 02 Schematic Design Total: | 6.25 | $1,562.50 |
|  |  |  | Additional Services Sub Total: | 6.25 | $1,562.50 |

### Table 5

|  |  |  |
| --- | --- | --- |
|  | Invoice Total: | $11,812.50 |
|  | Previous Balance: | -$26,588.00 |
| Payments Received: |  | $63,950.00 |
|  | Account Balance: | $11,812.50 |
|  |  |  |

In [4]:
from IPython.display import JSON, display

doc = result.documents[0] if result.documents else None

def compact(field):
    if not field:
        return None
    return {
        "content": field.get("content"),
        "confidence": field.get("confidence"),
        "type": field.get("type"),
    }

summary_fields = [
    "VendorName",
    "CustomerName",
    "InvoiceId",
    "InvoiceDate",
    "DueDate",
    "PurchaseOrder",
    "SubTotal",
    "TotalTax",
    "InvoiceTotal",
    "AmountDue",
]

summary = {}
items = []

if doc:
    summary = {
        name: compact(doc.fields.get(name))
        for name in summary_fields
        if doc.fields.get(name)
    }

    items_field = doc.fields.get("Items")
    for item in (items_field or {}).get("valueArray", []):
        value_object = item.get("valueObject", {})
        items.append({
            name: compact(value_object.get(name))
            for name in ["Description", "Quantity", "UnitPrice", "Amount"]
            if value_object.get(name)
        })

display(JSON({"summary": summary, "items": items}, expanded=True))

<IPython.core.display.JSON object>